# Т-поколение ИИ | Домашнее задание 1, МЛ профиль | Baseline


In [1]:
# Импорт нужных библиотек

import pandas as pd
import numpy as np
import time

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# Чтение данных

df_train = pd.read_csv('cardio_train.csv').astype('float32')
X, y = df_train.drop(columns=['cardio']).values, df_train['cardio'].values
X_test = pd.read_csv('cardio_test.csv').values.astype('float32')

In [61]:
df_train.cardio.value_counts()

,count
cardio,
0.0,8909
1.0,8850


In [29]:
# Константы

DEVICE = 'cuda'
BATCH_SIZE = 64

In [67]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sk = StandardScaler()
X = sk.fit_transform(X)
X_test = sk.transform(X_test)

Xt, Xv, yt, yv = train_test_split(X, y, stratify=y)

train_ds = TensorDataset(torch.from_numpy(Xt), torch.from_numpy(yt))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(torch.from_numpy(Xv), torch.from_numpy(yv))
train_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=True)


In [69]:
EPOCHS = 20

model = nn.Sequential(
    nn.Linear(195, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Linear(256, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(64, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
)
model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device).float()

        optimizer.zero_grad()

        logits = model(x).squeeze(-1)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    mean_loss = np.mean(losses)
    return mean_loss

losses = []
for epoch in range(EPOCHS):
    loss = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE
    )
    print(f'EPOCH: {epoch} LOSS: {loss}')
    losses.append(loss)

EPOCH: 0 LOSS: 0.6906966081687382
EPOCH: 1 LOSS: 0.6482746916157859
EPOCH: 2 LOSS: 0.4440304300614766
EPOCH: 3 LOSS: 0.3453966213124139
EPOCH: 4 LOSS: 0.2963819816708565
EPOCH: 5 LOSS: 0.2699034242757729
EPOCH: 6 LOSS: 0.23964958350573268
EPOCH: 7 LOSS: 0.21988650666815893
EPOCH: 8 LOSS: 0.20664414763450623
EPOCH: 9 LOSS: 0.1944090247154236
EPOCH: 10 LOSS: 0.17486813669758183
EPOCH: 11 LOSS: 0.17226308791765144
EPOCH: 12 LOSS: 0.14598676807114055
EPOCH: 13 LOSS: 0.1405981328870569
EPOCH: 14 LOSS: 0.14339442859802928
EPOCH: 15 LOSS: 0.13978847973048686
EPOCH: 16 LOSS: 0.12060875653156213
EPOCH: 17 LOSS: 0.11940464651478189
EPOCH: 18 LOSS: 0.10509018299302884
EPOCH: 19 LOSS: 0.0969475791257407


In [70]:
model.eval()
X_test_torch = torch.from_numpy(X_test).to(DEVICE)
with torch.no_grad():
    outputs = model(torch.tensor(X_test_torch, dtype=torch.float32))

    probs = torch.sigmoid(outputs)

    predictions = (probs >= 0.5).int()

predictions = predictions.cpu().numpy().flatten()
probs = probs.cpu().numpy().flatten()

/tmp/ipykernel_1112/510326446.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  outputs = model(torch.tensor(X_test_torch, dtype=torch.float32))


In [71]:
submission = pd.DataFrame({'cardio': predictions})

In [72]:
submission.to_csv('submission_final.csv', index_label='id')